# Formula 1 Podium Prediction — Milestone 1

**Task:** for every driver in a race, predict whether they finish on the podium (`positionOrder <= 3`),
using only information available **after qualifying and before the race starts**.

**Dataset:** Formula 1 World Championship (1950–2024), Kaggle / Ergast — 14 relational CSV tables.

## Table of contents
0. Setup
1. Raw EDA (before cleaning)
2. Data Cleaning & Auditing
3. Post-cleaning EDA
4. Data-Engineering Questions (Q1, Q2, Q3)
5. Feature Selection & Engineering
6. Train / Validation / Test Split
7. Pre-processing (incl. preprocessing-order experiments)
8. Predictive Modeling (baseline, statistical models, shallow NN)
9. Feature-Group Ablation
10. Explainability (XAI)
11. Inference Function
12. Summary & Limitations

# 0. Setup

This section makes the notebook reproducible and **Run-All ready**:
- imports and library versions (so results can be reproduced),
- one global random seed,
- project-wide constants (target, split boundaries),
- loading all 14 raw tables into an **immutable** `raw` dictionary — all cleaning later happens on copies,
- primary/foreign key definitions used by the audit in Section 2.

## 0.1 Optional installs
`lime` is not always pre-installed on Kaggle. On Kaggle, turn **Internet ON** in the notebook settings for this cell to work.
The cell is safe to re-run: it only installs what is missing.

In [1]:
import importlib.util, subprocess, sys

for pkg in ["shap", "lime"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    else:
        print(f"{pkg} already installed")

shap already installed
lime already installed


## 0.2 Imports and versions

In [2]:
import os
import sys
import importlib
import random
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

def _version(name):
    try:
        return importlib.import_module(name).__version__
    except Exception:
        return "not installed"

for lib in ["numpy", "pandas", "matplotlib", "seaborn", "sklearn",
            "xgboost", "lightgbm", "tensorflow", "torch", "shap", "lime"]:
    print(f"{lib:<12} {_version(lib)}")
print(f"{'python':<12} {sys.version.split()[0]}")

numpy        2.1.3
pandas       2.3.3
matplotlib   3.10.0
seaborn      0.13.2
sklearn      1.6.1
xgboost      3.4.1
lightgbm     4.6.0
tensorflow   2.20.0
torch        2.11.0+cpu
shap         0.52.0
lime         not installed
python       3.13.15


## 0.3 Reproducibility and project constants
All randomness (sampling, model initialisation, NN training) is driven by one seed.
The split boundaries are temporal (justified in Section 6): training on the past, evaluating on the future,
which mirrors how the model would be used in practice.

In [3]:
SEED = 42

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
    except ImportError:
        pass

set_seed()

# Target definition
TARGET = "podium"           # 1 if positionOrder <= 3 else 0
PODIUM_CUTOFF = 3

# Temporal split boundaries (seasons, inclusive)
TRAIN_END = 2019
VAL_YEARS = (2020, 2021)
TEST_START = 2022

# Era boundaries used in the data-engineering questions
HYBRID_ERA = (2014, 2021)
GROUND_EFFECT_ERA = (2022, 2024)

# Sentinel used for missing values in the raw CSVs (handled in Section 2)
SENTINEL = r"\N"

## 0.4 Locate the data
Finds the dataset automatically on Kaggle (`/kaggle/input/...`) or in a local `data/` folder,
so the same notebook runs in both places without manual edits.

In [4]:
TABLES = [
    "circuits", "constructors", "constructor_results", "constructor_standings",
    "drivers", "driver_standings", "races", "results", "qualifying",
    "lap_times", "pit_stops", "sprint_results", "seasons", "status",
]

def find_data_dir() -> Path:
    candidates = [Path("/kaggle/input"), Path("data"), Path("../data"), Path(".")]
    for root in candidates:
        if not root.exists():
            continue
        for hit in root.rglob("circuits.csv"):
            return hit.parent
    raise FileNotFoundError(
        "Could not find the F1 CSVs. On Kaggle, add the dataset via 'Add Input'; "
        "locally, place the 14 CSV files in a 'data/' folder."
    )

DATA_DIR = find_data_dir()
print("Data directory:", DATA_DIR)

missing_files = [t for t in TABLES if not (DATA_DIR / f"{t}.csv").exists()]
assert not missing_files, f"Missing tables: {missing_files}"
print(f"All {len(TABLES)} tables found.")

Data directory: /kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020
All 14 tables found.


## 0.5 Load raw tables (immutable)
The raw tables are loaded **as-is**: the `\N` sentinel is deliberately kept as a string here,
so the raw EDA in Section 1 can measure how much of it exists before we clean it.

To guarantee the raw data is never modified, each table gets a fingerprint (hash) at load time.
`get_table()` always returns a copy, and `assert_raw_unchanged()` can be called at any point to prove the raw data is intact.

In [5]:
raw = {t: pd.read_csv(DATA_DIR / f"{t}.csv", keep_default_na=False, na_values=[""])
       for t in TABLES}

def fingerprint(df: pd.DataFrame) -> str:
    return hashlib.md5(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()

RAW_FINGERPRINTS = {t: fingerprint(df) for t, df in raw.items()}

def get_table(name: str) -> pd.DataFrame:
    # Always work on a copy so the raw tables stay untouched.
    return raw[name].copy()

def assert_raw_unchanged() -> None:
    changed = [t for t, df in raw.items() if fingerprint(df) != RAW_FINGERPRINTS[t]]
    assert not changed, f"Raw tables were modified: {changed}"
    print("Raw tables unchanged ✔")

assert_raw_unchanged()

Raw tables unchanged ✔


## 0.6 Load sanity check
A first look at what was loaded: size of each table, how many `\N` sentinels it contains,
and which seasons the data covers. Detailed EDA follows in Section 1.

In [6]:
def count_sentinels(df: pd.DataFrame) -> int:
    return int((df.astype(str) == SENTINEL).sum().sum())

overview = pd.DataFrame({
    "rows": {t: len(df) for t, df in raw.items()},
    "columns": {t: df.shape[1] for t, df in raw.items()},
    "sentinel_cells": {t: count_sentinels(df) for t, df in raw.items()},
}).sort_values("rows", ascending=False)
overview["sentinel_%"] = (100 * overview["sentinel_cells"]
                          / (overview["rows"] * overview["columns"])).round(2)
display(overview)

years = raw["races"]["year"]
print(f"Seasons covered: {years.min()}–{years.max()}  ({years.nunique()} seasons, {len(raw['races'])} races)")
results_years = raw["results"].merge(raw["races"][["raceId", "year"]], on="raceId")["year"]
print(f"Seasons with results: {results_years.min()}–{results_years.max()}")

,rows,columns,sentinel_cells,sentinel_%
lap_times,589081,6,0,0.00
driver_standings,34863,7,0,0.00
results,26759,18,122887,25.51
constructor_standings,13391,7,0,0.00
constructor_results,12625,5,12608,19.97
pit_stops,11371,7,0,0.00
qualifying,10494,9,11600,12.28
races,1125,18,11349,56.04
drivers,861,9,1559,20.12
sprint_results,360,16,73,1.27


Seasons covered: 1950–2024  (75 seasons, 1125 races)
Seasons with results: 1950–2024


In [7]:
for t in TABLES:
    print(f"{t:<22} {list(raw[t].columns)}")

circuits               ['circuitId', 'circuitRef', 'name', 'location', 'country', 'lat', 'lng', 'alt', 'url']
constructors           ['constructorId', 'constructorRef', 'name', 'nationality', 'url']
constructor_results    ['constructorResultsId', 'raceId', 'constructorId', 'points', 'status']
constructor_standings  ['constructorStandingsId', 'raceId', 'constructorId', 'points', 'position', 'positionText', 'wins']
drivers                ['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob', 'nationality', 'url']
driver_standings       ['driverStandingsId', 'raceId', 'driverId', 'points', 'position', 'positionText', 'wins']
races                  ['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time']
results                ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'posi

## 0.7 Key definitions (used by the audit in Section 2)
Primary keys identify each row; foreign keys must point to an existing primary key in the referenced table.
Defining them once here keeps the audit in Section 2 short and consistent.
Composite keys (e.g. `lap_times`) are tables whose rows are only unique for a combination of columns.

In [8]:
PRIMARY_KEYS = {
    "circuits": ["circuitId"],
    "constructors": ["constructorId"],
    "constructor_results": ["constructorResultsId"],
    "constructor_standings": ["constructorStandingsId"],
    "drivers": ["driverId"],
    "driver_standings": ["driverStandingsId"],
    "races": ["raceId"],
    "results": ["resultId"],
    "qualifying": ["qualifyId"],
    "lap_times": ["raceId", "driverId", "lap"],
    "pit_stops": ["raceId", "driverId", "stop"],
    "sprint_results": ["resultId"],
    "seasons": ["year"],
    "status": ["statusId"],
}

# (child table, child column, parent table, parent column)
FOREIGN_KEYS = [
    ("races", "circuitId", "circuits", "circuitId"),
    ("races", "year", "seasons", "year"),
    ("results", "raceId", "races", "raceId"),
    ("results", "driverId", "drivers", "driverId"),
    ("results", "constructorId", "constructors", "constructorId"),
    ("results", "statusId", "status", "statusId"),
    ("qualifying", "raceId", "races", "raceId"),
    ("qualifying", "driverId", "drivers", "driverId"),
    ("qualifying", "constructorId", "constructors", "constructorId"),
    ("sprint_results", "raceId", "races", "raceId"),
    ("sprint_results", "driverId", "drivers", "driverId"),
    ("sprint_results", "constructorId", "constructors", "constructorId"),
    ("sprint_results", "statusId", "status", "statusId"),
    ("lap_times", "raceId", "races", "raceId"),
    ("lap_times", "driverId", "drivers", "driverId"),
    ("pit_stops", "raceId", "races", "raceId"),
    ("pit_stops", "driverId", "drivers", "driverId"),
    ("driver_standings", "raceId", "races", "raceId"),
    ("driver_standings", "driverId", "drivers", "driverId"),
    ("constructor_standings", "raceId", "races", "raceId"),
    ("constructor_standings", "constructorId", "constructors", "constructorId"),
    ("constructor_results", "raceId", "races", "raceId"),
    ("constructor_results", "constructorId", "constructors", "constructorId"),
]

# Sanity check: every key column actually exists in the loaded tables
for t, cols in PRIMARY_KEYS.items():
    assert set(cols) <= set(raw[t].columns), f"PK columns missing in {t}: {cols}"
for child, ccol, parent, pcol in FOREIGN_KEYS:
    assert ccol in raw[child].columns and pcol in raw[parent].columns, (child, ccol, parent, pcol)
print(f"{len(PRIMARY_KEYS)} primary keys and {len(FOREIGN_KEYS)} foreign keys defined ✔")

14 primary keys and 23 foreign keys defined ✔


## 0.8 Columns that must never be features (leakage guard)
These columns are only known **after** the race. They are listed here once so later sections can assert
that none of them ends up in the feature matrix. The full leakage audit (with reasons per column) is in Section 5.

In [9]:
POST_RACE_COLUMNS = {
    # results.csv
    "position", "positionText", "positionOrder", "points", "laps", "time",
    "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed", "statusId",
    # derived target
    TARGET,
}
POST_RACE_TABLES = {"lap_times", "pit_stops"}   # current-race data, never features
# Standings tables are post-race *for the current race*; they are only usable after shifting
# to "standings before this race" (done in Section 5).

def assert_no_leakage(feature_columns) -> None:
    leaked = set(feature_columns) & POST_RACE_COLUMNS
    assert not leaked, f"Post-race columns in features: {sorted(leaked)}"
    print("No post-race columns in features ✔")

# 1. Raw EDA (before cleaning)

_Goal: measure data-quality issues with plots and statistics before changing anything (sentinels per column, races and entrants per season, podium base rate per season, qualifying coverage per season, duplicate (raceId, driverId) pairs, status frequencies)._

In [10]:
# TODO

# 2. Data Cleaning & Auditing

_Sentinel handling, semantic typing (dates, lap times → seconds), primary/foreign key audit, duplicate rule for shared drives, scope decisions (Indy 500, DNQ rows, grid = 0)._

In [11]:
# TODO

# 3. Post-cleaning EDA

_Re-run the key Section 1 plots on the cleaned data and show the before/after impact._

In [12]:
# TODO

# 4. Data-Engineering Questions

_Q1: front-row → podium conversion by circuit, pre vs post-2014 (grid vs qualifying).  
Q2: home-race effect controlling for starting position.  
Q3: mechanical-retirement rate by constructor, 2014–2021 vs 2022–2024._

In [13]:
# TODO

# 5. Feature Selection & Engineering

_Leakage-safe grid, qualifying, standings (shifted), form (past races only) and context features, each justified with evidence; full leakage audit table._

In [14]:
# TODO

# 6. Train / Validation / Test Split

_Temporal split: train ≤ 2019, validation 2020–2021, test ≥ 2022 — with justification._

In [15]:
# TODO

# 7. Pre-processing

_Imputation, scaling and encoding fit on training data only; effect on distributions; at least two preprocessing-order experiments._

In [16]:
# TODO

# 8. Predictive Modeling

_Baseline, at least two statistical ML models, shallow NN; ROC-AUC, PR-AUC, F1 on train/validation/test; thresholds and hyperparameters chosen on validation only._

In [17]:
# TODO

# 9. Feature-Group Ablation

_Remove one feature group at a time, retrain, and report the performance drop._

In [18]:
# TODO

# 10. Explainability (XAI)

_Global: permutation importance and SHAP. Local: SHAP waterfall and LIME. Importance is not causation._

In [19]:
# TODO

# 11. Inference Function

_Predict for a single driver-race record; demonstrated on a complete and a partially-missing record._

In [20]:
# TODO

# 12. Summary & Limitations

_Key results, chosen primary/secondary metric, and limitations (qualifying coverage drift, era sensitivity)._

In [21]:
# TODO

In [22]:
assert_raw_unchanged()  # final check: raw data was never modified

Raw tables unchanged ✔
